# 03. 전처리 및 프롬프트 최적화 (Preprocessing)

**목표**: 검증된 corpus를 NLP 분석에 적합한 형태로 변환

| 단계 | 내용 |
|------|------|
| 텍스트 정제 | 특수문자, 공백, 이모지 제거 |
| 프롬프트 최적화 | 구어체 정규화, 쿼리 품질 개선 |
| 형태소 분석 | KoNLPy Okt — 명사/동사/형용사 추출 |
| 불용어 제거 | 반려견 도메인 불용어 사전 적용 |
| 토큰화 결과 저장 | 매칭 실험용 토큰 컬럼 추가 |

> **다음 단계**: `04_eda.ipynb` — 생애주기별 질병 빈도 분석 및 시각화

## 0. 환경 설정

In [1]:
import sys
sys.path.append('..')

import re
import pandas as pd
from konlpy.tag import Okt
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from utils.config import DATA_PROCESSED

VALIDATED_PATH     = DATA_PROCESSED / 'corpus_validated.csv'
PREPROCESSED_PATH  = DATA_PROCESSED / 'corpus_preprocessed.csv'

okt = Okt()
print("Okt 초기화 완료")

Okt 초기화 완료


## 1. 데이터 로드

In [2]:
df = pd.read_csv(VALIDATED_PATH)
print(f"로드 완료: {len(df):,}개")
df.head(2)

로드 완료: 21,604개


,lifeCycle,department,disease,input,output,split
0,성견,안과,제3안검탈출증,저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 강아지에 대...,해당 증상은 제3안검이 라고 불리는 부분과 관련이 있습니다. 이는 정상적인 해부학적...,train
1,성견,안과,기타,저희 강아지가 어제 밤부터 한쪽 눈을 잘 뜨지 못하는 모습을 보였습니다. 눈을 살펴...,눈에 관련된 문제라 많은 걱정을 하시게 되실 것입니다. 눈을 제대로 뜨지 못하는 증...,train


## 2. 텍스트 정제

In [3]:
def clean_text(text: str) -> str:
    """특수문자, 이모지, 과도한 공백 제거 후 소문자 정규화."""
    if not isinstance(text, str):
        return ''
    # 이모지 및 특수문자 제거 (한글, 영문, 숫자, 기본 문장부호 유지)
    text = re.sub(r'[^\w\s가-힣ㄱ-ㅎㅏ-ㅣ.,?!]', ' ', text)
    # 연속 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 정제 전후 샘플 비교
sample = df['input'].iloc[0]
print(f"[정제 전]\n{sample[:200]}")
print(f"\n[정제 후]\n{clean_text(sample)[:200]}")

[정제 전]
저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 강아지에 대해 문의드립니다. 최근 몇 일간 눈의 흰자 부분이 약간 붉어졌는데, 오늘 자세히 살펴보니 눈의 상태가 매우 이상하게 변해 있었습니다. 이러한 증상은 무엇을 의미하는 것인지 궁금합니다. 항염 외용제 사용에서 중요한 포인트를 알려 주실 수 있을까요 간단히 알려 주실 수 있을까요?

[정제 후]
저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 강아지에 대해 문의드립니다. 최근 몇 일간 눈의 흰자 부분이 약간 붉어졌는데, 오늘 자세히 살펴보니 눈의 상태가 매우 이상하게 변해 있었습니다. 이러한 증상은 무엇을 의미하는 것인지 궁금합니다. 항염 외용제 사용에서 중요한 포인트를 알려 주실 수 있을까요 간단히 알려 주실 수 있을까요?


In [4]:
df['input_clean']  = df['input'].apply(clean_text)
df['output_clean'] = df['output'].apply(clean_text)
print("텍스트 정제 완료")

텍스트 정제 완료


## 3. 프롬프트 최적화 (쿼리 정규화)

사용자가 입력하는 구어체 쿼리를 매칭 성능에 유리한 형태로 정규화.

In [5]:
# 구어체 → 표준어 매핑 사전
COLLOQUIAL_MAP = {
    '강아지': '반려견',
    '댕댕이': '반려견',
    '멍멍이': '반려견',
    '토해요': '구토',
    '토했어요': '구토',
    '안 먹어요': '식욕부진',
    '안먹어요': '식욕부진',
    '눈물': '눈 분비물',
    '절뚝': '파행',
    '긁어요': '소양감',
    '긁는다': '소양감',
}

def normalize_query(text: str) -> str:
    """구어체를 표준 의학 용어로 치환하여 매칭 정확도 향상."""
    for colloquial, standard in COLLOQUIAL_MAP.items():
        text = text.replace(colloquial, standard)
    return text

# 적용
df['input_normalized'] = df['input_clean'].apply(normalize_query)

# 변환 확인
changed = (df['input_clean'] != df['input_normalized']).sum()
print(f"구어체 정규화 적용: {changed:,}개 ({changed/len(df)*100:.1f}%)")

구어체 정규화 적용: 15,480개 (71.7%)


## 4. 불용어 사전 정의

In [6]:
# 반려견 도메인 불용어 (매칭에 도움이 되지 않는 단어)
STOPWORDS = {
    # 일반 불용어
    '것', '수', '등', '및', '또', '더', '이', '그', '저', '때',
    '해', '할', '하는', '있는', '없는', '되는', '같은',
    # 질문 패턴 불용어
    '궁금', '문의', '알고싶', '알려주', '부탁', '감사',
    '어떻게', '어떤', '무엇', '왜', '언제', '어디',
    # 수의사 답변 패턴 불용어
    '말씀', '드리', '경우', '통해', '위해', '대해',
    '생각', '판단', '확인', '진행', '필요'
}

print(f"불용어 사전 크기: {len(STOPWORDS)}개")

불용어 사전 크기: 40개


## 5. 형태소 분석 (KoNLPy Okt) — 병렬 처리

In [7]:
# 추출 품사: 명사(Noun), 동사(Verb), 형용사(Adjective)
TARGET_POS = {'Noun', 'Verb', 'Adjective'}

def tokenize(text: str) -> str:
    """형태소 분석 후 불용어 제거, 공백 구분 토큰 문자열 반환."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    tokens = [
        word for word, pos in okt.pos(text, stem=True)
        if pos in TARGET_POS
        and word not in STOPWORDS
        and len(word) > 1
    ]
    return ' '.join(tokens)

# 샘플 확인
sample = df['input_normalized'].iloc[0]
print(f"[원문]\n{sample[:150]}")
print(f"\n[토큰화 결과]\n{tokenize(sample)}")

[원문]
저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 반려견에 대해 문의드립니다. 최근 몇 일간 눈의 흰자 부분이 약간 붉어졌는데, 오늘 자세히 살펴보니 눈의 상태가 매우 이상하게 변해 있었습니다. 이러한 증상은 무엇을 의미하는 것인지 궁금합니다. 항

[토큰화 결과]
저희 기르다 있다 되다 암컷 말티즈 다른 견종 혼합 반려견 드리다 최근 일간 흰자 부분 약간 붉다 오늘 자세하다 살펴보다 상태 매우 이상하다 변하다 있다 이러하다 증상 의미 하다 궁금하다 항염 용제 사용 중요하다 포인트 알다 줄다 있다 간단하다 알다 줄다 있다 환경 관리 시작 하다 흔하다 하다 실수 알다 줄다 있다 요점 위주 설명 하다 줄다 있다


In [8]:
# Okt는 thread-safe하지 않으므로 순차 처리
# (ProcessPoolExecutor는 konlpy와 충돌 가능 — 순차가 안전)
tqdm.pandas(desc="형태소 분석")
df['input_tokens']  = df['input_normalized'].progress_apply(tokenize)
df['output_tokens'] = df['output_clean'].progress_apply(tokenize)

print("형태소 분석 완료")

형태소 분석: 100%|██████████| 21604/21604 [03:32<00:00, 101.77it/s]

형태소 분석 완료


## 6. 전처리 결과 확인

In [9]:
# 토큰이 비어있는 행 확인
empty_tokens = (df['input_tokens'].str.len() == 0).sum()
print(f"토큰 없는 행: {empty_tokens}개")

# 평균 토큰 수
df['token_count'] = df['input_tokens'].apply(lambda x: len(x.split()) if x else 0)
print(f"평균 토큰 수: {df['token_count'].mean():.1f}개")
print(f"최소: {df['token_count'].min()} / 최대: {df['token_count'].max()}")

토큰 없는 행: 0개
평균 토큰 수: 74.0개
최소: 5 / 최대: 818


In [10]:
# 전처리 파이프라인 전후 비교 샘플
idx = 0
print("=== 전처리 파이프라인 전후 비교 ===")
print(f"[원문]\n{df['input'].iloc[idx][:200]}")
print(f"\n[정제 후]\n{df['input_clean'].iloc[idx][:200]}")
print(f"\n[정규화 후]\n{df['input_normalized'].iloc[idx][:200]}")
print(f"\n[토큰화 후]\n{df['input_tokens'].iloc[idx]}")

=== 전처리 파이프라인 전후 비교 ===
[원문]
저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 강아지에 대해 문의드립니다. 최근 몇 일간 눈의 흰자 부분이 약간 붉어졌는데, 오늘 자세히 살펴보니 눈의 상태가 매우 이상하게 변해 있었습니다. 이러한 증상은 무엇을 의미하는 것인지 궁금합니다. 항염 외용제 사용에서 중요한 포인트를 알려 주실 수 있을까요 간단히 알려 주실 수 있을까요?

[정제 후]
저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 강아지에 대해 문의드립니다. 최근 몇 일간 눈의 흰자 부분이 약간 붉어졌는데, 오늘 자세히 살펴보니 눈의 상태가 매우 이상하게 변해 있었습니다. 이러한 증상은 무엇을 의미하는 것인지 궁금합니다. 항염 외용제 사용에서 중요한 포인트를 알려 주실 수 있을까요 간단히 알려 주실 수 있을까요?

[정규화 후]
저희 집에서 기르고 있는 것으로 7세 된 암컷 말티즈와 다른 견종 혼합 반려견에 대해 문의드립니다. 최근 몇 일간 눈의 흰자 부분이 약간 붉어졌는데, 오늘 자세히 살펴보니 눈의 상태가 매우 이상하게 변해 있었습니다. 이러한 증상은 무엇을 의미하는 것인지 궁금합니다. 항염 외용제 사용에서 중요한 포인트를 알려 주실 수 있을까요 간단히 알려 주실 수 있을까요?

[토큰화 후]
저희 기르다 있다 되다 암컷 말티즈 다른 견종 혼합 반려견 드리다 최근 일간 흰자 부분 약간 붉다 오늘 자세하다 살펴보다 상태 매우 이상하다 변하다 있다 이러하다 증상 의미 하다 궁금하다 항염 용제 사용 중요하다 포인트 알다 줄다 있다 간단하다 알다 줄다 있다 환경 관리 시작 하다 흔하다 하다 실수 알다 줄다 있다 요점 위주 설명 하다 줄다 있다


## 7. 저장

In [11]:
# 불필요 컬럼 제거 후 저장
df_out = df.drop(columns=['token_count'])
df_out.to_csv(PREPROCESSED_PATH, index=False, encoding='utf-8-sig')

print(f"저장 완료: {PREPROCESSED_PATH}")
print(f"최종 컬럼: {list(df_out.columns)}")
print(f"레코드 수: {len(df_out):,}개")

저장 완료: /Users/user/Desktop/project/pet-health-ai/notebooks/../data/processed/corpus_preprocessed.csv
최종 컬럼: ['lifeCycle', 'department', 'disease', 'input', 'output', 'split', 'input_clean', 'output_clean', 'input_normalized', 'input_tokens', 'output_tokens']
레코드 수: 21,604개


## 8. 전처리 요약

| 단계 | 처리 내용 | 결과 |
|------|----------|------|
| 텍스트 정제 | 특수문자·이모지 제거 | `input_clean` |
| 프롬프트 최적화 | 구어체 → 표준 의학 용어 | `input_normalized` |
| 형태소 분석 | Okt 명사/동사/형용사 추출 | `input_tokens` |
| 불용어 제거 | 도메인 불용어 사전 적용 | (tokenize 내 처리) |
| 저장 파일 | - | `data/processed/corpus_preprocessed.csv` |

> **다음 단계**: `04_eda.ipynb` — 생애주기별 질병 빈도 분석 및 워드클라우드